# Chapter 2 gallery — location & scale

Reproduces `flour.R` (Table 2.4) from the RobStatTM example scripts. Every numeric result is cross-checked bit-for-bit against direct R.

In [1]:
import os, sys, pathlib

# Windows R_HOME setup (skip if already configured)
if sys.platform == "win32" and "R_HOME" not in os.environ:
    os.environ["R_HOME"] = r"C:\Program Files\R\R-4.5.2"
    os.environ["PATH"] = r"C:\Program Files\R\R-4.5.2\bin\x64;" + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe for CI execution
import matplotlib.pyplot as plt
import robstatm_py as rpm
from robstatm_py import set_seed
from robstatm_py._r import r as _r

ro = _r()
ro.r("suppressMessages(library(RobStatTM))")
FIG_DIR = pathlib.Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print(f"robstatm_py {rpm.__version__}")

robstatm_py 0.0.1.dev0


## flour — bisquare location M-estimator (Table 2.4)

`flour.R` compares the sample mean, the bisquare M-estimator (`locScaleM`, efficiency 0.95) and the 25% trimmed mean on the flour aflatoxin data (n=24), with 95% confidence intervals.

In [2]:
flour = rpm.datasets.flour()
x = flour.iloc[:, 0].to_numpy(dtype=float)
n = len(x); qn = 1.959963984540054  # qnorm(0.975)

res = rpm.loc_scale_m(x, eff=0.95)
muM, muMst = res.mu, res.std_mu
interM = (muM - muMst * qn, muM + muMst * qn)

xbar = x.mean(); smed = x.std(ddof=1) / np.sqrt(n)
inter_mean = (xbar - smed * qn, xbar + smed * qn)

print(f'sample mean      = {xbar:.4f}   CI = ({inter_mean[0]:.4f}, {inter_mean[1]:.4f})')
print(f'bisquare M-est   = {muM:.4f}   CI = ({interM[0]:.4f}, {interM[1]:.4f})')
print(f'robust dispersion (scale) = {res.disper:.4f}')

sample mean      = 4.2804   CI = (2.1611, 6.3998)
bisquare M-est   = 3.1168   CI = (2.8961, 3.3375)
robust dispersion (scale) = 0.6947


### Strict-tier cross-check vs direct R `locScaleM`

In [3]:
ro.globalenv['xf'] = x
ro.r('rf <- locScaleM(xf, eff=0.95)')
print('mu      bit-equal to R:', muM == float(ro.r('rf$mu')[0]))
print('std.mu  bit-equal to R:', muMst == float(ro.r('rf$std.mu')[0]))
print('disper  bit-equal to R:', res.disper == float(ro.r('rf$disper')[0]))

mu      bit-equal to R: True
std.mu  bit-equal to R: True
disper  bit-equal to R: True
